In [2]:
import polars as pl

# 1. Define schemas
msg_columns = ["time", "event_type", "order_id", "size", "price", "direction"]
ob_columns = []
for i in range(1, 11): 
    ob_columns.extend([f"Ask_Price_{i}", f"Ask_Size_{i}", f"Bid_Price_{i}", f"Bid_Size_{i}"])

# 2. Load the data (REPLACE WITH YOUR EXACT FILE NAMES!)
msg_file = "data/LOBSTER_SampleFile_AMZN_2012-06-21_10/AMZN_2012-06-21_34200000_57600000_message_10.csv" 
ob_file = "data/LOBSTER_SampleFile_AMZN_2012-06-21_10/AMZN_2012-06-21_34200000_57600000_orderbook_10.csv" 

df_messages = pl.read_csv(msg_file, has_header=False, new_columns=msg_columns)
df_orderbook = pl.read_csv(ob_file, has_header=False, new_columns=ob_columns)

# 3. Glue them together horizontally
df_market = pl.concat([df_messages, df_orderbook], how="horizontal")

# 4. Calculate Spread and Mid-Price in dollars
df_market = df_market.with_columns([
    ((pl.col("Ask_Price_1") - pl.col("Bid_Price_1")) / 10000).alias("Spread_Dollars"),
    (((pl.col("Ask_Price_1") + pl.col("Bid_Price_1")) / 2) / 10000).alias("Mid_Price_Dollars")
])

# 5. View the results
print(df_market.select(["time", "event_type", "Spread_Dollars", "Mid_Price_Dollars"]).head(10))

shape: (10, 4)
┌──────────────┬────────────┬────────────────┬───────────────────┐
│ time         ┆ event_type ┆ Spread_Dollars ┆ Mid_Price_Dollars │
│ ---          ┆ ---        ┆ ---            ┆ ---               │
│ f64          ┆ i64        ┆ f64            ┆ f64               │
╞══════════════╪════════════╪════════════════╪═══════════════════╡
│ 34200.01746  ┆ 5          ┆ 0.77           ┆ 223.565           │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 

/tmp/ipykernel_14347/1376694075.py:17: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  df_market = pl.concat([df_messages, df_orderbook], how="horizontal")


In [3]:
# Calculate Volume Imbalance at Level 1
df_market = df_market.with_columns([
    (pl.col("Bid_Size_1") / (pl.col("Bid_Size_1") + pl.col("Ask_Size_1"))).alias("Imbalance_Ratio")
])

# Let's look at the time, the sizes, and our new ratio
columns_to_view = ["time", "Bid_Size_1", "Ask_Size_1", "Imbalance_Ratio"]
print(df_market.select(columns_to_view).head(15))

shape: (15, 4)
┌──────────────┬────────────┬────────────┬─────────────────┐
│ time         ┆ Bid_Size_1 ┆ Ask_Size_1 ┆ Imbalance_Ratio │
│ ---          ┆ ---        ┆ ---        ┆ ---             │
│ f64          ┆ i64        ┆ i64        ┆ f64             │
╞══════════════╪════════════╪════════════╪═════════════════╡
│ 34200.01746  ┆ 100        ┆ 100        ┆ 0.5             │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
│ …            ┆ …          ┆ …          ┆ …               │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
│ 34200.189608 ┆ 21         ┆ 100        ┆ 0.173554        │
└────────

In [4]:
# Calculate Micro-Price in dollars
df_market = df_market.with_columns([
    (
        (pl.col("Bid_Price_1") * pl.col("Ask_Size_1") + pl.col("Ask_Price_1") * pl.col("Bid_Size_1")) 
        / (pl.col("Bid_Size_1") + pl.col("Ask_Size_1")) 
        / 10000 # Convert to standard dollars
    ).alias("Micro_Price_Dollars")
])

# Let's compare the standard Mid-Price to our new Micro-Price
columns_to_view = ["time", "Mid_Price_Dollars", "Micro_Price_Dollars", "Imbalance_Ratio"]
print(df_market.select(columns_to_view).head(15))

shape: (15, 4)
┌──────────────┬───────────────────┬─────────────────────┬─────────────────┐
│ time         ┆ Mid_Price_Dollars ┆ Micro_Price_Dollars ┆ Imbalance_Ratio │
│ ---          ┆ ---               ┆ ---                 ┆ ---             │
│ f64          ┆ f64               ┆ f64                 ┆ f64             │
╞══════════════╪═══════════════════╪═════════════════════╪═════════════════╡
│ 34200.01746  ┆ 223.565           ┆ 223.565             ┆ 0.5             │
│ 34200.189608 ┆ 223.88            ┆ 223.834298          ┆ 0.173554        │
│ 34200.189608 ┆ 223.88            ┆ 223.834298          ┆ 0.173554        │
│ 34200.189608 ┆ 223.88            ┆ 223.834298          ┆ 0.173554        │
│ 34200.189608 ┆ 223.88            ┆ 223.834298          ┆ 0.173554        │
│ …            ┆ …                 ┆ …                   ┆ …               │
│ 34200.189608 ┆ 223.88            ┆ 223.834298          ┆ 0.173554        │
│ 34200.189608 ┆ 223.88            ┆ 223.834298          ┆ 0.

In [5]:
# Generate the list of column names for all 10 levels
bid_size_cols = [f"Bid_Size_{i}" for i in range(1, 11)]
ask_size_cols = [f"Ask_Size_{i}" for i in range(1, 11)]

# Calculate total volume across all 10 levels
df_market = df_market.with_columns([
    pl.sum_horizontal(bid_size_cols).alias("Cumulative_Bid_Size"),
    pl.sum_horizontal(ask_size_cols).alias("Cumulative_Ask_Size")
])

# Calculate the Deep Imbalance Ratio (using all 10 levels)
df_market = df_market.with_columns([
    (pl.col("Cumulative_Bid_Size") / (pl.col("Cumulative_Bid_Size") + pl.col("Cumulative_Ask_Size"))).alias("Deep_Imbalance_Ratio")
])

# Let's compare the Level 1 Imbalance vs the Deep Imbalance
columns_to_view = ["time", "Imbalance_Ratio", "Deep_Imbalance_Ratio"]
print(df_market.select(columns_to_view).head(15))

shape: (15, 3)
┌──────────────┬─────────────────┬──────────────────────┐
│ time         ┆ Imbalance_Ratio ┆ Deep_Imbalance_Ratio │
│ ---          ┆ ---             ┆ ---                  │
│ f64          ┆ f64             ┆ f64                  │
╞══════════════╪═════════════════╪══════════════════════╡
│ 34200.01746  ┆ 0.5             ┆ 0.862266             │
│ 34200.189608 ┆ 0.173554        ┆ 0.861303             │
│ 34200.189608 ┆ 0.173554        ┆ 0.867445             │
│ 34200.189608 ┆ 0.173554        ┆ 0.867445             │
│ 34200.189608 ┆ 0.173554        ┆ 0.866441             │
│ …            ┆ …               ┆ …                    │
│ 34200.189608 ┆ 0.173554        ┆ 0.316681             │
│ 34200.189608 ┆ 0.173554        ┆ 0.301903             │
│ 34200.189608 ┆ 0.173554        ┆ 0.295512             │
│ 34200.189608 ┆ 0.173554        ┆ 0.283509             │
│ 34200.189608 ┆ 0.173554        ┆ 0.28282              │
└──────────────┴─────────────────┴──────────────────────┘

In [6]:
# Define our prediction horizon (20 events/ticks into the future)
horizon = 20

# 1. Pull the future Mid-Price into the current row
df_market = df_market.with_columns([
    pl.col("Mid_Price_Dollars").shift(-horizon).alias("Future_Mid_Price")
])

# 2. Compare future price to current price to create the ML label
df_market = df_market.with_columns([
    pl.when(pl.col("Future_Mid_Price") > pl.col("Mid_Price_Dollars")).then(1)
    .when(pl.col("Future_Mid_Price") < pl.col("Mid_Price_Dollars")).then(-1)
    .otherwise(0)
    .alias("Target_Direction")
])

# Let's view our new ML-ready dataset!
# (We use drop_nulls() to temporarily hide the very last 20 rows of the day, which have no future)
columns_to_view = ["time", "Mid_Price_Dollars", "Future_Mid_Price", "Deep_Imbalance_Ratio", "Target_Direction"]
print(df_market.select(columns_to_view).drop_nulls().head(15))

shape: (15, 5)
┌──────────────┬───────────────────┬──────────────────┬──────────────────────┬──────────────────┐
│ time         ┆ Mid_Price_Dollars ┆ Future_Mid_Price ┆ Deep_Imbalance_Ratio ┆ Target_Direction │
│ ---          ┆ ---               ┆ ---              ┆ ---                  ┆ ---              │
│ f64          ┆ f64               ┆ f64              ┆ f64                  ┆ i32              │
╞══════════════╪═══════════════════╪══════════════════╪══════════════════════╪══════════════════╡
│ 34200.01746  ┆ 223.565           ┆ 223.88           ┆ 0.862266             ┆ 1                │
│ 34200.189608 ┆ 223.88            ┆ 223.88           ┆ 0.861303             ┆ 0                │
│ 34200.189608 ┆ 223.88            ┆ 223.88           ┆ 0.867445             ┆ 0                │
│ 34200.189608 ┆ 223.88            ┆ 223.88           ┆ 0.867445             ┆ 0                │
│ 34200.189608 ┆ 223.88            ┆ 223.88           ┆ 0.866441             ┆ 0                │
│ …  

In [ ]:
# Save the engineered dataframe to a Parquet file for our ML models to use later
output_path = "data/AMZN_engineered_features.parquet"
df_market.write_parquet(output_path)

print(f"Success! Data saved to {output_path}")